In [ ]:
# The chooser inputs unpacked as plain directories, not zips, and the puzzle
# images live under a layout that has to be discovered rather than assumed --
# the repo's own kaggle notebook learned that the hard way, so its search is
# reused here.
import os, sys, shutil
from pathlib import Path

BASE = Path("/kaggle/input")
# Kaggle mounts under /kaggle/input/datasets/<owner>/<slug>, not under the slug
# directly, so both searches have to walk rather than list one level.
print("mounted:", [q.name for q in BASE.iterdir()])
IN = next((q for q in BASE.rglob("*")
           if q.is_dir() and (q / "src").is_dir() and (q / "top5").is_dir()),
          None)
assert IN is not None, "chooser inputs not found under " + str(
    [str(q) for q in BASE.rglob("top5")])
print("chooser inputs:", IN)
W = Path("/kaggle/working")
WORK = W / "pazzle_work"
(WORK / "cache").mkdir(parents=True, exist_ok=True)
(WORK / "ckpt").mkdir(parents=True, exist_ok=True)

SRC = IN / "src"
assert SRC.is_dir(), f"no src in {list(IN.iterdir())}"
sys.path.insert(0, str(SRC))
for f in (IN / "ckpt").glob("*.pt"):
    shutil.copy(f, WORK / "ckpt" / f.name)
shutil.copy(IN / "restore_labels.npz", WORK / "cache" / "restore_labels.npz")

TOP5 = IN / "top5"
print("src files:", len(list(SRC.glob("*.py"))), " top5 dumps:",
      len(list(TOP5.glob("*.npz"))))


def find_data():
    """A directory holding train/inputs, wherever Kaggle chose to mount it."""
    for cand in sorted(Path("/kaggle/input").rglob("inputs")):
        if cand.is_dir() and cand.parent.name == "train":
            return cand.parent.parent
    return None


DATA = find_data()
print("DATA:", DATA)
assert DATA is not None, [p.name for p in Path("/kaggle/input").iterdir()]
print("train inputs:", len(list((DATA / "train" / "inputs").glob("*.png"))))
os.environ["PAZZLE_DATA"] = str(DATA)
os.environ["PAZZLE_WORK"] = str(WORK)
import torch
print("cuda:", torch.cuda.is_available())


In [ ]:
# Top-5 candidate dumps, from BOTH regions, with the control built in.
#
# The matchers were trained on names[:-300], so their scores on those boards
# are sharper than anything they will see at test time. A chooser trained
# there would learn the wrong distribution. The held-out 300 are the honest
# region and 160 of them are already dumped, so this fills the remaining 140
# and then takes a large batch from the training region -- and REPORTS the
# top-1 accuracy of each, so the shift is measured rather than assumed.
import numpy as np, time
from pathlib import Path
from config import CACHE_DIR, CKPT_DIR, GRID as G, TRAIN_INP
from restore_tile import to_frags
from seam_cost import costs_from_models
from seam_embed import SeamEmbed
import cv2, torch

N, K, dev = G * G, 5, "cuda"
OUT = Path("/kaggle/working/top5_new"); OUT.mkdir(exist_ok=True)

def rgb(p):
    return np.ascontiguousarray(cv2.imread(str(p), cv2.IMREAD_COLOR)[:, :, ::-1])

def load(n):
    c = torch.load(Path(CKPT_DIR) / n, map_location=dev, weights_only=False)
    a = c["args"]
    m = SeamEmbed(a["ch"], a["blocks"], a["dim"], a["strip"],
                  a.get("head", "global"),
                  predict=any(k.startswith("pred.") for k in c["model"])).to(dev)
    m.load_state_dict(c["model"]); m.modes = a.get("modes", 1); m.eval()
    return m

ms = [load(n) for n in ("seam_embed_v3.pt", "seam_embed_local.pt",
                        "seam_embed_wide.pt")]
blob = np.load(Path(CACHE_DIR) / "restore_labels.npz", allow_pickle=True)
NAMES = [str(x) for x in blob["names"]]
INV = blob["inv"]

def dump_one(gi, tag):
    tiles = to_frags(rgb(f"{TRAIN_INP}/{NAMES[gi]}")).astype(np.float32)[
        INV[gi].astype(np.int64)]
    CH, CV = costs_from_models(ms, tiles)
    store, top1, tot = {}, 0, 0
    for M, t, step, ok in ((-np.asarray(CH, np.float64), "h", 1,
                            lambda i: i % G != G - 1),
                           (-np.asarray(CV, np.float64), "v", G,
                            lambda i: i < N - G)):
        D = np.array(M); np.fill_diagonal(D, -1e9)
        idx = np.argpartition(-D, K, axis=1)[:, :K]
        val = np.take_along_axis(D, idx, axis=1)
        o = np.argsort(-val, axis=1)
        idx = np.take_along_axis(idx, o, 1); val = np.take_along_axis(val, o, 1)
        lab = np.full(N, K, np.int8)
        for i in range(N):
            if not ok(i):
                lab[i] = -1; continue
            tot += 1
            hit = np.where(idx[i] == i + step)[0]
            if len(hit): lab[i] = int(hit[0])
            top1 += int(idx[i, 0] == i + step)
        store[f"{t}_idx"] = idx.astype(np.int16)
        store[f"{t}_val"] = val.astype(np.float32)
        store[f"{t}_lab"] = lab
    np.savez_compressed(OUT / f"{tag}_{gi:05d}.npz", name=NAMES[gi],
                        inv=INV[gi].astype(np.int32), **store)
    return top1, tot

BUDGET = 8.0 * 3600
t0 = time.time()
stats = {"held": [0, 0, 0], "train": [0, 0, 0]}
plan = ([(len(NAMES) - 300 + i, "held") for i in range(160, 300)]
        + [(i, "train") for i in range(0, 3000)])
for gi, tag in plan:
    if time.time() - t0 > BUDGET:
        print("budget reached"); break
    a, b = dump_one(gi, tag)
    s = stats[tag]; s[0] += a; s[1] += b; s[2] += 1
    if s[2] % 25 == 0:
        el = time.time() - t0
        print(f"{tag}: {s[2]} boards, top-1 {s[0]/max(s[1],1):.4f}, "
              f"{el/60:.0f} min elapsed", flush=True)
print("\nTHE CONTROL that decides how these may be used:")
for tag, s in stats.items():
    if s[2]:
        print(f"  {tag:>5}: {s[2]:4d} boards, matcher top-1 {s[0]/max(s[1],1):.4f}")
print("if the two differ, only the held-out region may train the chooser")
